# Divergent behavioural selector holdout

This is the remaining code-backed critical experiment. It runs held-out selector comparisons under predeclared constraint-frontier configurations, rather than only the dominant baseline configuration. Each configuration selects on seeds 60--62 and evaluates frozen selectors on disjoint seeds 260--269.


In [ ]:
from pathlib import Path
import sys,json
import pandas as pd
C=[Path.cwd(),Path.cwd()/'paper-ideas'/'CURE-Rec'/'code',*Path.cwd().parents]
ROOT=next(p for p in C if (p/'pyproject.toml').exists() and (p/'cure_rec').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from cure_rec.config import load_settings
from cure_rec.revision import run_selector_holdout_study
RUN_ALL=True
CONFIG=ROOT/'configs'/'curesim_full.yaml'
OUT=ROOT/'results'/'reviewer_phase_assets'/'divergent_behavioral_holdout'
FRONTIER=[('strict_provider',0.24,-0.08,0.65),('tight_relevance',0.28,-0.03,0.65),('relaxed_provider',0.34,-0.08,0.65)]
OUT.mkdir(parents=True,exist_ok=True)
print('Root:',ROOT)

In [ ]:
if RUN_ALL:
    runs=[]
    for name,provider,relevance,fatigue in FRONTIER:
        cfg=load_settings(CONFIG); cfg.constraints.max_provider_disparity=provider; cfg.constraints.min_relevance_delta=relevance; cfg.constraints.max_fatigue=fatigue; cfg.run.output_root=ROOT/'runs'/'divergent-behavioral'
        run_dir=run_selector_holdout_study(cfg, selection_seeds=(60,61,62), evaluation_seeds=tuple(range(260,270)))
        runs.append(str(run_dir)); print(name,run_dir)
    (OUT/'run_manifest.json').write_text(json.dumps({'frontier':FRONTIER,'selection_seeds':[60,61,62],'evaluation_seeds':list(range(260,270)),'runs':runs,'claim_scope':'held-out CURE-Sim behavioural selector comparison; not real causal inference'},indent=2))
else: print('Disabled.')

In [ ]:
# Aggregate all generated selector summaries for manuscript inspection
frames=[]
run_root=ROOT/'runs'/'divergent-behavioral'
for p in run_root.glob('**/heldout_selector_summary.csv'):
    x=pd.read_csv(p)
    x.insert(0,'run',p.parent.relative_to(ROOT).as_posix())
    frames.append(x)
if frames:
    table=pd.concat(frames,ignore_index=True)
    table.to_csv(OUT/'divergent_selector_summary.csv',index=False)
    print('Wrote',OUT/'divergent_selector_summary.csv','rows',len(table))
    display(table)
else:
    print('No completed selector summaries found under',run_root)
